# Activity 1: Explore code and run reference simulation

## Import libraries

In [1]:
from pathlib import Path

# Standard library imports
import os
import sys
import matplotlib
import pandas as pd
from matplotlib import pyplot as plt

# Hydrobiogeochemical model and utilities

utils_path = os.path.abspath("../src/")# Add the directory containing script.py to the Python path
if utils_path not in sys.path:
    sys.path.append(utils_path)

from devcon2026.hydrology import (
    Hydrology,
    HydrologyArtifactNames,
    HydrologyParameters,
    HydrologyStates,
)
from devcon2026.hydrology.export import (
    convert_fluxes_to_nitrogen_units,
    convert_states_to_nitrogen_units,
)
from devcon2026.nitrogen import Nitrogen, NitrogenParameters, NitrogenThreeCompartment
from devcon2026.tables import read_table, write_table
from devcon2026.utils import apply_time_window, run_hydrology_scenario, add_channel_concentrations, run_nitrogen_scenario, plot_hydrology_scenarios, plot_forcing_scenarios, plot_concentration_scenarios, plot_mass_scenarios

## Define data source and output directories

In [2]:
# User defined
NAME_SCENARIO_BASE = "activity_1"
FORCING_SCENARIO = "fallspring" # Options "fallspring" or "spring" 
WITH_ADSORPTION = True # Set to True to include adsorption in the nitrogen model, False to exclude it. Note that this will affect the parameters used in the nitrogen model and the name of the scenario, so make sure to change those accordingly.

if WITH_ADSORPTION:
    NAME_SCENARIO = f"{NAME_SCENARIO_BASE}_{FORCING_SCENARIO}_adsorption" # Change this as a function of the processes included in the scenario and remember to change the parameters acordingly
else:
    NAME_SCENARIO = f"{NAME_SCENARIO_BASE}_{FORCING_SCENARIO}_no_adsorption" # Change this as a function of the processes included in the scenario and remember to change the parameters acordingly

FORCE_HYDROLOGY = True # Set to True to force the hydrology model with the hydrology forcing data, False to use the hydrology model output as forcing for the nitrogen model
FORCE_NITROGEN = True # Set to True to force the nitrogen model with the nitrogen forcing data, False to use the nitrogen model output as forcing for the next time step (i.e., fully coupled mode)
SHOW_PROGRESS = True # Set to True to show progress bars during model execution, False to hide them. Note that this will not affect the progress bars shown by the hydrology and nitrogen models, which are controlled by the `show_progress` argument in the `run` method of each model.

# Output directory and input paths
OUTPUT_DIR = Path(f"demo_outputs_{NAME_SCENARIO}") 
HYDROLOGY_FORCING_PARQUET = Path("../data/hydrology_forcings.parquet")
NITROGEN_FORCING_PARQUET = Path(f"../data/nitrogen_forcing_{FORCING_SCENARIO}.parquet")

# Output paths
HYDROLOGY_OUTPUT_DIR = OUTPUT_DIR / "hydrology_3layer"
SCENARIO_OUTPUT_DIR = OUTPUT_DIR / "nitrogen_3layer_scenarios"

HYDROLOGY_FORCINGS_PLOT = OUTPUT_DIR / "hydrology_3layer_variants.png"
FORCINGS_PLOT = OUTPUT_DIR / "hydrology_nitrogen_forcings.png"
NITROGEN_DIN_PLOT = OUTPUT_DIR / "nitrogen_3layer_din_scenarios.png"
NITROGEN_DON_PLOT = OUTPUT_DIR / "nitrogen_3layer_don_scenarios.png"
NITROGEN_DIN_MASS_PLOT = OUTPUT_DIR / "nitrogen_3layer_din_mass_scenarios.png"
NITROGEN_DON_MASS_PLOT = OUTPUT_DIR / "nitrogen_3layer_don_mass_scenarios.png"

HYDROLOGY_ARTIFACTS = HydrologyArtifactNames(
    discharge="discharge1.parquet",
    states="states1.parquet",
    fluxes="fluxes1.parquet",
    forcing="model_ready_hydrology_forcing.parquet",
)


## Define spin-up and simulation period

Data is available from 01-01-2007 to 12-31-2017

In [3]:
# HYDROLOGY_SPIN_START = "2007-01-01"
# NITROGEN_SPIN_START = "2008-01-01"
# RESULTS_START = "2013-01-01"
# SIMULATION_END = "2018-01-01"

HYDROLOGY_SPIN_START = "2007-01-01"
NITROGEN_SPIN_START = "2008-01-01"
RESULTS_START = "2009-01-01"
SIMULATION_END = "2010-01-01"

## Model initial states and parameters

### Hydrologic model

In [4]:
HYDROLOGY_INITIAL_STATES = HydrologyStates(
    s_sn=0.01,  # snow storage [m]
    s_s=0.03,  # soil storage [m]
    s_gwa=0.2,  # active GW storage [m]
    s_gwp=0.5,  # passive GW storage [m]
)

HYDROLOGY_PARAMS = HydrologyParameters(
    t_0=0.0,  # snow-rain temperature threshold [C]
    m_sn=0.002,  # snowmelt storage scale [m]
    k_sn=1.157e-7,  # degree-day snowmelt coefficient [m/s/C]
    c_e=0.8,  # actual ET correction coefficient [1]
    s_max=0.1,  # maximum soil storage [m]
    m_s=1e-5,  # soil ET shape parameter [1]
    beta_s=2.0,  # soil wetness exponent [1]
    k_sgw=1e-7,  # vertical groundwater recharge flux scale [m/s]
    k_gwpc=1e-7,  # passive groundwater recession coefficient [1/s]
    k_gwap=1e-6,  # active-to-passive groundwater transfer coefficient [1/s]
    k_gwac=1e-6,  # active groundwater channel flux scale [m/s]
    beta_gwac=2.0,  # active groundwater channel exponent [1]
    s_gwa_max=1.0,  # maximum active groundwater storage [m]
    s_ref_td=0.005,  # legacy relative tile drainage activation threshold [1]
    tile_drainage_method="relative_storage",  # tile drainage formulation [method]
    k_td=1e-4,  # tile drainage coefficient [1/s]
    gamma_x=-0.34,  # seasonal infiltration partition offset [1]
    gamma_i=0.32,  # seasonal infiltration partition amplitude [1]
    gamma_p=336.0,  # seasonal infiltration partition phase [day]
    pet_albedo=0.23,  # surface albedo for reference ET [1]
    pet_emissivity=0.98,  # surface emissivity for reference ET [1]
    n_gu=2.0,  # gamma unit hydrograph shape parameter [1]
    a_gu_seconds=2 * 60 * 60,  # gamma unit hydrograph scale parameter [s]
    area_km2=100.0,  # drainage area [km2]
)

## Biogeochemical model

In [5]:
NITROGEN_PARAMS = NitrogenParameters(
    s_wp=20.0,  # wilting point soil water storage [mm]
    s_max=1 * HYDROLOGY_PARAMS.s_max,  # maximum soil water storage [mm]
    min_dissolved_storage=1.0,  # minimum storage for dissolved concentrations [mm]
    smf_sat=0.8,  # saturated moisture factor [1]
    beta_sm=1.0,  # moisture factor exponent [1]
    rel_saturation_low=0.2,  # low relative saturation threshold [1]
    rel_saturation_high=0.9,  # high relative saturation threshold [1]
    rel_sat_limit_exp=0.7,  # exponential moisture limitation threshold [1]
    beta_exp=2.5,  # exponential moisture factor exponent [1]
    v_degrad_son=1e-5,  # maximum degradation rate of slow organic nitrogen [1/day]
    v_dissol_son=1e-5,  # maximum dissolution rate of slow organic nitrogen [1/day]
    v_dissol_fon=1e-3,  # maximum dissolution rate of fast organic nitrogen [1/day]
    v_min_fon=1e-3,  # maximum mineralization rate of fast organic nitrogen [1/day]
    v_denit=5e-2,  # maximum denitrification rate [1/day]
    k_denit=1.5,  # denitrification half-saturation concentration [mg/L]
    uptake_demand=50.0,  # plant inorganic nitrogen uptake demand [kg N/km2/day]
    delta_time_solver=1.0 / 24.0,  # nitrogen solver time step [day]
    freundlich_exponent=1.0,  # Freundlich DON adsorption exponent [1]
    freundlich_constant=100.0,  # Freundlich DON adsorption constant [(mg N/kg soil)/(mg N/L)^n]
    soil_bulk_density=1.3,  # soil bulk density used for DON adsorption [kg/L]
    deposition_din_fraction=1.0,
    deposition_don_fraction=0.0,
    deposition_son_fraction=0.0,
    deposition_fon_fraction=0.0,
    fertilizer_din_fraction=0.0,
    fertilizer_don_fraction=0.0,
    fertilizer_son_fraction=0.0,
    fertilizer_fon_fraction=1.0,
    manure_din_fraction=0.0,
    manure_don_fraction=0.0,
    manure_son_fraction=0.0,
    manure_fon_fraction=1.0,
)

NITROGEN_SOIL_PARAMS = NITROGEN_PARAMS
NITROGEN_GWA_PARAMS = NITROGEN_PARAMS
NITROGEN_GWP_PARAMS = NITROGEN_PARAMS.with_updates(
    {"uptake_demand": 0.0, "v_denit": NITROGEN_PARAMS.v_denit / 10.0}
)

In [6]:
utils_params = {
    "FORCING_SCENARIO": FORCING_SCENARIO,
    "NAME_SCENARIO": NAME_SCENARIO,
    "FORCE_HYDROLOGY": FORCE_HYDROLOGY,
    "FORCE_NITROGEN": FORCE_NITROGEN,
    "SHOW_PROGRESS": SHOW_PROGRESS,
    "OUTPUT_DIR": OUTPUT_DIR,
    "HYDROLOGY_FORCING_PARQUET": HYDROLOGY_FORCING_PARQUET,
    "NITROGEN_FORCING_PARQUET": NITROGEN_FORCING_PARQUET,
    "HYDROLOGY_OUTPUT_DIR": HYDROLOGY_OUTPUT_DIR,
    "SCENARIO_OUTPUT_DIR": SCENARIO_OUTPUT_DIR,
    "HYDROLOGY_FORCINGS_PLOT": HYDROLOGY_FORCINGS_PLOT,
    "FORCINGS_PLOT": FORCINGS_PLOT,
    "NITROGEN_DIN_PLOT": NITROGEN_DIN_PLOT,
    "NITROGEN_DON_PLOT": NITROGEN_DON_PLOT,
    "NITROGEN_DIN_MASS_PLOT": NITROGEN_DIN_MASS_PLOT,
    "NITROGEN_DON_MASS_PLOT": NITROGEN_DON_MASS_PLOT,
    "HYDROLOGY_ARTIFACTS": HYDROLOGY_ARTIFACTS,
    "HYDROLOGY_INITIAL_STATES": HYDROLOGY_INITIAL_STATES,
    "HYDROLOGY_PARAMS": HYDROLOGY_PARAMS,
    "NITROGEN_PARAMS": NITROGEN_PARAMS,
    "NITROGEN_SOIL_PARAMS": NITROGEN_SOIL_PARAMS,
    "NITROGEN_GWA_PARAMS": NITROGEN_GWA_PARAMS,
    "NITROGEN_GWP_PARAMS": NITROGEN_GWP_PARAMS,
    "HYDROLOGY_SPIN_START": HYDROLOGY_SPIN_START,
    "NITROGEN_SPIN_START": NITROGEN_SPIN_START,
    "RESULTS_START": RESULTS_START,
    "SIMULATION_END": SIMULATION_END,
}

## Run models

In [7]:
OUTPUT_DIR.mkdir(exist_ok=True)
SCENARIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

hydrology_name = "Hydrologic_model"
nitrogen_name = "Nitrogen_model"
# Run hydrologic model
(states, fluxes, meteorology) = run_hydrology_scenario(
    name = hydrology_name, 
    params = HYDROLOGY_PARAMS, 
    utils_params=utils_params
)

# Run biogeochemical model 
solution_nitrogen = run_nitrogen_scenario(
                hydrology_name,
                nitrogen_name,
                states,
                fluxes,
                meteorology,
                with_soil_don_adsorption=WITH_ADSORPTION,
                utils_params=utils_params
)
               
print(f"Saved scenario outputs to {SCENARIO_OUTPUT_DIR}")

Running hydrology scenario: Hydrologic_model


hydrology:   0%|          | 0/26304 [00:00<?, ?step/s]

Hydrology scenario Hydrologic_model: generated from ../data/hydrology_forcings.parquet
Starting nitrogen scenario: Hydrologic_model_Nitrogen_model
Running nitrogen scenario: Hydrologic_model_Nitrogen_model


nitrogen Hydrologic_model_Nitrogen_model:   0%|          | 0/17543 [00:00<?, ?step/s]

Saved nitrogen scenario: Hydrologic_model_Nitrogen_model
Saved scenario outputs to demo_outputs_activity_1_fallspring_adsorption/nitrogen_3layer_scenarios


## Plot the results

In [8]:
hydrology_outputs = {"Hydro_results": (states, fluxes, meteorology)}
nitrogen_outputs = {"Nitrogen_results": solution_nitrogen}

plot_hydrology_scenarios(hydrology_outputs, utils_params=utils_params)
plot_forcing_scenarios(hydrology_outputs, utils_params=utils_params)

plot_concentration_scenarios(
    nitrogen_outputs, species="din", output_path=NITROGEN_DIN_PLOT, utils_params=utils_params
)

plot_concentration_scenarios(
    nitrogen_outputs, species="don", output_path=NITROGEN_DON_PLOT, utils_params=utils_params
)

plot_mass_scenarios(nitrogen_outputs, species="din", output_path=NITROGEN_DIN_MASS_PLOT, utils_params=utils_params)
plot_mass_scenarios(nitrogen_outputs, species="don", output_path=NITROGEN_DON_MASS_PLOT, utils_params=utils_params)
